# 数据读取

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
from datasets import load_from_disk
# 读取音频嵌入数据
embeddings_path = "/content/drive/MyDrive/yambda_500m/embeddings"
embeddings = load_from_disk(embeddings_path)

Loading dataset from disk:   0%|          | 0/32 [00:00<?, ?it/s]

In [7]:
# 读取用户物品交互数据
multi_event_path = "/content/drive/MyDrive/yambda_500m/multi_event"
multi_event = load_from_disk(multi_event_path)

Loading dataset from disk:   0%|          | 0/29 [00:00<?, ?it/s]

In [8]:
# 读取艺人专辑歌曲关系图
album_item_mapping_path = "/content/drive/MyDrive/yambda_500m/album_item_mapping"
album_item_mapping = load_from_disk(album_item_mapping_path)
artist_item_mapping_path = "/content/drive/MyDrive/yambda_500m/artist_item_mapping"
artist_item_mapping = load_from_disk(artist_item_mapping_path)

In [9]:
import torch

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 是否可用: {torch.cuda.is_available()}")

PyTorch 版本: 2.10.0+cu128
CUDA 是否可用: True


In [10]:
# 🔥 开启 TF32 加速模式 (A100 显卡必备)
# 'high' 表示允许使用 TensorFloat32，精度略微下降但速度飞快
torch.set_float32_matmul_precision('high')

# 数据预处理

In [11]:
import pandas as pd
import numpy as np
import torch
import time
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

start_time = time.time()

print("🚀 第一步：极速加载数据到内存...")
# 只提取我们需要的列，避免加载无用数据
cols_to_keep = ['uid', 'item_id', 'timestamp', 'is_organic', 'event_type', 'played_ratio_pct']

# 利用底层的 PyArrow 引擎，瞬间转为 Pandas DataFrame
df = multi_event.select_columns(cols_to_keep).to_pandas()

print("🧹 第二步：正在进行高标准的数据清洗...")
# 瞬间填充空值为 0
df['played_ratio_pct'] = df['played_ratio_pct'].fillna(0.0)

# Pandas 向量化条件过滤，这在底层是 C 语言运行的，只需几秒钟
mask_like = df['event_type'] == 'like'
mask_listen = (df['event_type'] == 'listen') & (df['played_ratio_pct'] >= 50.0)

valid_mask = (mask_like | mask_listen)

total_raw = len(df)
# 应用掩码，保留干净数据
df_clean = df[valid_mask]
total_valid = len(df_clean)

# 修改后：保护长尾物品的单向过滤
print("🧹 正在进行【保留长尾】的高标准清洗...")

# 1. 物品端：极轻度过滤（比如只过滤掉全站只听过 1 次的绝对噪音，保留 >=2 的所有长尾）
min_item_inter = 2
item_counts = df_clean.groupby('item_id').size()
valid_i = item_counts[item_counts >= min_item_inter].index
df_clean = df_clean[df_clean['item_id'].isin(valid_i)]

# 2. 用户端：严格过滤（保证每个用户至少有 5 条记录，才能切分出 训练集/测试集）
min_user_inter = 5
user_counts = df_clean.groupby('uid').size()
valid_u = user_counts[user_counts >= min_user_inter].index
df_clean = df_clean[df_clean['uid'].isin(valid_u)]

print(f"   -> 原始交互总数: {total_raw}")
print(f"   -> 清洗后有效交互数: {total_valid} (剔除了 {total_raw - total_valid} 条无效交互)")

print("⏳ 第三步：提取纯 C 级底层数组并进行全局排序...")
# 直接提取 DataFrame 的 .values (底层就是连续的 NumPy C 数组)
uids = df_clean['uid'].values
item_ids = df_clean['item_id'].values
timestamps = df_clean['timestamp'].values

# 联合排序 (按 uid 排，uid 内部按 timestamp 排)
sort_idx = np.lexsort((timestamps, uids))
sorted_uids = uids[sort_idx]
sorted_items = item_ids[sort_idx]

print("⚡ 第四步：极速切分用户序列...")
# 核心加速魔法：利用 numpy 的 C 级数组切分
change_indices = np.where(sorted_uids[:-1] != sorted_uids[1:])[0] + 1
unique_uids = np.concatenate(([sorted_uids[0]], sorted_uids[change_indices]))
split_items = np.split(sorted_items, change_indices)

print("📦 第五步：正在组装用户序列字典...")
# 组装字典
user_seqs = {
    u.item(): items.tolist()
    for u, items in tqdm(zip(unique_uids, split_items), total=len(unique_uids), desc="组装进度", unit="user")
}

print(f"✅ 序列构建完成！总耗时: {time.time() - start_time:.2f} 秒\n")


🚀 第一步：极速加载数据到内存...
🧹 第二步：正在进行高标准的数据清洗...
🧹 正在进行【保留长尾】的高标准清洗...
   -> 原始交互总数: 480255564
   -> 清洗后有效交互数: 304024436 (剔除了 176231128 条无效交互)
⏳ 第三步：提取纯 C 级底层数组并进行全局排序...
⚡ 第四步：极速切分用户序列...
📦 第五步：正在组装用户序列字典...


组装进度: 100%|██████████| 98929/98929 [00:17<00:00, 5575.74user/s]

✅ 序列构建完成！总耗时: 256.94 秒



In [12]:
# ==========================================
# ✂️ 划分数据集与构建 PyTorch Dataset
# ==========================================
train_seqs, val_seqs, test_seqs = {}, {}, {}

for uid, seq in tqdm(user_seqs.items(), desc="划分数据集", unit="user"):
    if len(seq) < 3:
        continue
    train_seqs[uid] = seq[:-2]
    val_seqs[uid] = seq[:-1]
    test_seqs[uid] = seq

print(f"\n✅ 有效训练用户数: {len(train_seqs)}")

class SequenceDataset(Dataset):
    def __init__(self, user_seqs, max_len=50):
        self.uids = list(user_seqs.keys())
        self.seqs = [user_seqs[uid] for uid in self.uids]
        self.max_len = max_len

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, idx):
        seq = self.seqs[idx]
        seq = seq[-self.max_len-1:]

        pad_len = self.max_len + 1 - len(seq)
        seq = [0] * pad_len + seq

        x = torch.tensor(seq[:-1], dtype=torch.long)
        y = torch.tensor(seq[1:], dtype=torch.long)
        return x, y

MAX_SEQ_LEN = 50
train_dataset = SequenceDataset(train_seqs, max_len=MAX_SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

划分数据集: 100%|██████████| 98929/98929 [00:12<00:00, 8064.49user/s]


✅ 有效训练用户数: 98929


In [13]:
print("🚀 正在构建全局对齐的声学特征矩阵...")
start_emb_time = time.time()

# 1. 将 embeddings 数据集极速转为 Pandas DataFrame
df_emb = embeddings.select_columns(['item_id', 'embed']).to_pandas()

# 2. 找到全局最大的 item_id，以确定我们需要多大的词表 (Vocabulary Size)
# 确保矩阵的行数足够容纳所有的 item_id
max_item_id = max(df_clean['item_id'].max(), df_emb['item_id'].max())
feature_dim = len(df_emb.iloc[0]['embed'])

# 3. 初始化全 0 矩阵 (行数为 max_item_id + 1，第 0 行自然被保留为 0 用于 Padding)
track_vectors = np.zeros((max_item_id + 1, feature_dim), dtype=np.float32)

# 4. 利用 NumPy 的高级索引 (Advanced Indexing) 瞬间将特征填入对应行
# df_emb['item_id'].values 作为行索引，精准对齐
track_vectors[df_emb['item_id'].values] = np.vstack(df_emb['embed'].values)

num_tracks = max_item_id # 这将是我们传入模型的物品总量

print(f"✅ 特征矩阵构建完成！")
print(f"   -> 矩阵形状: {track_vectors.shape} (行数=词表大小+1, 列数=特征维度)")
print(f"   -> 总耗时: {time.time() - start_emb_time:.2f} 秒\n")

🚀 正在构建全局对齐的声学特征矩阵...
✅ 特征矩阵构建完成！
   -> 矩阵形状: (9390624, 128) (行数=词表大小+1, 列数=特征维度)
   -> 总耗时: 71.80 秒



# 定义模型

## id序列模型

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class FDSA_ID_Only(nn.Module):
    def __init__(self, num_items, item_dim=64, max_seq_len=50, num_heads=4, num_layers=2, dropout=0.2):
        super(FDSA_ID_Only, self).__init__()
        self.max_seq_len = max_seq_len

        # 1. 只有 ID Embedding，没有任何多模态特征
        self.item_emb = nn.Embedding(num_items + 1, item_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, item_dim)

        # 2. 单流 Transformer (仅处理 ID)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=item_dim, nhead=num_heads,
            dim_feedforward=item_dim * 4, dropout=dropout, batch_first=True
        )
        self.id_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

    def forward(self, seqs):
        device = seqs.device
        batch_size, seq_len = seqs.size()

        padding_mask = (seqs == 0)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=device).bool()
        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
        pos_embeddings = self.pos_emb(positions)

        # 纯 ID 流前向传播
        id_embs = self.item_emb(seqs) + pos_embeddings
        id_out = self.id_encoder(self.dropout(id_embs), mask=causal_mask, src_key_padding_mask=padding_mask)

        # 直接输出归一化隐状态
        return F.normalize(id_out, p=2, dim=-1, eps=1e-8)

    def get_item_representation(self, item_ids):
        # 物品表征仅由 ID 构成
        e_id = self.item_emb(item_ids)
        return F.normalize(e_id, p=2, dim=-1, eps=1e-8)

# 初始化纯 ID 模型
model_id_only = FDSA_ID_Only(
    num_items=num_tracks,
    item_dim=64,
    num_heads=4,
    num_layers=2
).to(device)

print("✅ FDSA_ID_Only (w/o Acoustic) 定义并初始化完毕！")

✅ FDSA_ID_Only (w/o Acoustic) 定义并初始化完毕！


## id+音频双流模型

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SharedGatedFusion(nn.Module):
    """
    共享门控融合头：
    - sequence 侧：融合上下文化后的 ID / acoustic 表示
    - item 侧：融合静态 ID / acoustic 表示
    两侧共享同一组 gate/proj 参数，保证打分发生在同一表示空间。
    """
    def __init__(self, hidden_dim, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gate = nn.Linear(hidden_dim * 2, 2)
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def compute_gate(self, x_id, x_feat):
        concat = torch.cat([x_id, x_feat], dim=-1)           # [..., 2H]
        gate = torch.softmax(self.gate(concat), dim=-1)      # [..., 2]
        return gate

    def forward(self, x_id, x_feat, return_gate=False):
        gate = self.compute_gate(x_id, x_feat)
        fused = gate[..., 0:1] * x_id + gate[..., 1:2] * x_feat
        fused = torch.relu(self.proj(self.dropout(fused)))
        fused = F.normalize(fused, p=2, dim=-1, eps=1e-8)
        if return_gate:
            return fused, gate
        return fused


class FDSA_Final(nn.Module):
    def __init__(
        self,
        num_items,
        item_dim=64,
        feature_dim=128,
        max_seq_len=50,
        pretrained_features=None,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super(FDSA_Final, self).__init__()
        self.max_seq_len = max_seq_len
        self.item_dim = item_dim

        # 1. ID 与音频 Embedding (0 设为 Padding)
        self.item_emb = nn.Embedding(num_items + 1, item_dim, padding_idx=0)
        self.feature_emb = nn.Embedding(num_items + 1, feature_dim, padding_idx=0)

        if pretrained_features is not None:
            feat_weight = torch.tensor(pretrained_features, dtype=torch.float32)
            if feat_weight.size(0) == num_items:
                # 若外部特征不含 padding 行，则自动前补零
                feat_weight = torch.cat(
                    [torch.zeros(1, feat_weight.size(1), dtype=feat_weight.dtype), feat_weight],
                    dim=0
                )
            if feat_weight.size(0) != num_items + 1:
                raise ValueError(
                    f"pretrained_features 行数应为 {num_items + 1}（含 padding）或 {num_items}（不含 padding），"
                    f"当前为 {feat_weight.size(0)}"
                )
            self.feature_emb.weight.data.copy_(feat_weight)
            self.feature_emb.weight.requires_grad = True

        self.feature_proj = nn.Linear(feature_dim, item_dim)
        self.pos_emb = nn.Embedding(max_seq_len, item_dim)

        # 2. 双流 Transformer
        id_encoder_layer = nn.TransformerEncoderLayer(
            d_model=item_dim, nhead=num_heads,
            dim_feedforward=item_dim * 4, dropout=dropout, batch_first=True
        )
        feat_encoder_layer = nn.TransformerEncoderLayer(
            d_model=item_dim, nhead=num_heads,
            dim_feedforward=item_dim * 4, dropout=dropout, batch_first=True
        )
        self.id_encoder = nn.TransformerEncoder(id_encoder_layer, num_layers=num_layers)
        self.feature_encoder = nn.TransformerEncoder(feat_encoder_layer, num_layers=num_layers)

        # 3. 共享融合头：sequence 侧和 item 侧共用
        self.fusion = SharedGatedFusion(item_dim, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def _build_masks(self, seqs):
        device = seqs.device
        batch_size, seq_len = seqs.size()
        padding_mask = (seqs == 0)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=device).bool()
        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, seq_len)
        return padding_mask, causal_mask, positions

    def encode_modalities(self, seqs):
        """
        返回上下文化后的两路表示：
        z_id, z_feat: [B, L, D]
        """
        batch_size, seq_len = seqs.size()
        padding_mask, causal_mask, positions = self._build_masks(seqs)
        pos_embeddings = self.pos_emb(positions)

        # 流 1: ID
        id_embs = self.item_emb(seqs) + pos_embeddings
        z_id = self.id_encoder(
            self.dropout(id_embs),
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        # 流 2: Acoustic Feature
        feat_embs = self.feature_proj(self.feature_emb(seqs)) + pos_embeddings
        z_feat = self.feature_encoder(
            self.dropout(feat_embs),
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        return z_id, z_feat

    def forward(self, seqs, return_gate=False):
        """
        序列编码：输出归一化后的隐状态 [B, L, D]
        """
        z_id, z_feat = self.encode_modalities(seqs)
        if return_gate:
            fused_out, gate = self.fusion(z_id, z_feat, return_gate=True)
            return fused_out, gate
        return self.fusion(z_id, z_feat, return_gate=False)

    def get_item_modalities(self, item_ids):
        e_id = self.item_emb(item_ids)
        e_feat = self.feature_proj(self.feature_emb(item_ids))
        return e_id, e_feat

    def get_item_representation(self, item_ids, return_gate=False):
        """
        物品编码：输出归一化后的物品表征 [..., D]
        注意：这里也走同一个共享融合头，而不是直接 e_id + e_feat
        """
        e_id, e_feat = self.get_item_modalities(item_ids)
        if return_gate:
            rep, gate = self.fusion(e_id, e_feat, return_gate=True)
            return rep, gate
        return self.fusion(e_id, e_feat, return_gate=False)

    def get_last_step_gate(self, seqs):
        """
        返回最后一步的 gate 权重，便于可视化：
        gates: [B, 2]，两列分别是 [ID_gate, Acoustic_gate]
        """
        _, gate = self.forward(seqs, return_gate=True)
        return gate[:, -1, :]

# 重新初始化模型
model = FDSA_Final(
    num_items=num_tracks,
    pretrained_features=track_vectors,
    item_dim=64,
    num_heads=4,
    num_layers=2
).to(device)

print("✅ FDSA_Final（共享门控融合头版）模型已定义并初始化完毕！")

✅ FDSA_Final（共享门控融合头版）模型已定义并初始化完毕！


## 图序列混合网络

In [25]:

import torch
import torch.nn as nn
import torch.nn.functional as F


class SharedGatedFusion(nn.Module):
    """共享门控融合头：sequence 侧与 item 侧共用同一套参数。"""

    def __init__(self, hidden_dim, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gate = nn.Linear(hidden_dim * 3, 3)
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def compute_gate(self, x_id, x_acoustic, x_graph):
        concat = torch.cat([x_id, x_acoustic, x_graph], dim=-1)
        gate = torch.softmax(self.gate(concat), dim=-1)
        return gate

    def forward(self, x_id, x_acoustic, x_graph, return_gate=False):
        # 允许输入形状为 [B, L, D] 或 [N, D]
        gate = self.compute_gate(x_id, x_acoustic, x_graph)

        fused = (
            gate[..., 0:1] * x_id
            + gate[..., 1:2] * x_acoustic
            + gate[..., 2:3] * x_graph
        )
        fused = self.proj(fused)
        fused = self.dropout(torch.relu(fused))
        fused = self.norm(fused)
        fused = F.normalize(fused, p=2, dim=-1, eps=1e-8)
        if return_gate:
            return fused, gate
        return fused


class FDSA_Graph_Fusion(nn.Module):
    def __init__(self, num_items, item_dim=64, acoustic_dim=128, graph_dim=64,
                 pretrained_acoustic=None, pretrained_graph=None,
                 max_seq_len=50, num_heads=4, num_layers=2, dropout=0.2):
        super(FDSA_Graph_Fusion, self).__init__()
        self.max_seq_len = max_seq_len
        self.item_dim = item_dim
        self.num_items = num_items

        # ==========================================
        # 1. 三大底层 Embedding 层
        # ==========================================
        self.item_emb = nn.Embedding(num_items + 1, item_dim, padding_idx=0)

        self.acoustic_emb = nn.Embedding(num_items + 1, acoustic_dim, padding_idx=0)
        self.graph_emb = nn.Embedding(num_items + 1, graph_dim, padding_idx=0)

        self._load_pretrained_embedding(self.acoustic_emb, pretrained_acoustic, acoustic_dim, 'acoustic')
        self._load_pretrained_embedding(self.graph_emb, pretrained_graph, graph_dim, 'graph')

        # 统一维度投影
        self.acoustic_proj = nn.Linear(acoustic_dim, item_dim)
        self.graph_proj = nn.Linear(graph_dim, item_dim)
        self.pos_emb = nn.Embedding(max_seq_len, item_dim)
        self.dropout = nn.Dropout(dropout)

        # ==========================================
        # 2. 三流独立 Transformer Encoders
        # ==========================================
        self.id_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=item_dim,
                nhead=num_heads,
                dim_feedforward=item_dim * 4,
                dropout=dropout,
                batch_first=True,
            ),
            num_layers=num_layers,
        )
        self.acoustic_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=item_dim,
                nhead=num_heads,
                dim_feedforward=item_dim * 4,
                dropout=dropout,
                batch_first=True,
            ),
            num_layers=num_layers,
        )
        self.graph_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=item_dim,
                nhead=num_heads,
                dim_feedforward=item_dim * 4,
                dropout=dropout,
                batch_first=True,
            ),
            num_layers=num_layers,
        )

        # ==========================================
        # 3. 共享融合头（关键改动）
        # ==========================================
        self.fusion = SharedGatedFusion(hidden_dim=item_dim, dropout=dropout)

    def _load_pretrained_embedding(self, emb_layer, pretrained_matrix, expected_dim, name):
        if pretrained_matrix is None:
            return

        weight = torch.tensor(pretrained_matrix, dtype=torch.float32)
        if weight.ndim != 2 or weight.size(1) != expected_dim:
            raise ValueError(
                f"{name} pretrained matrix shape mismatch: expected (*, {expected_dim}), got {tuple(weight.shape)}"
            )

        # 兼容两种输入：
        # 1) [num_items + 1, dim]（已含 padding 行）
        # 2) [num_items, dim]     （不含 padding 行，需要前面补零）
        if weight.size(0) == self.num_items + 1:
            emb_layer.weight.data.copy_(weight)
        elif weight.size(0) == self.num_items:
            padded = torch.zeros(self.num_items + 1, expected_dim, dtype=torch.float32)
            padded[1:] = weight
            emb_layer.weight.data.copy_(padded)
        else:
            raise ValueError(
                f"{name} pretrained matrix row mismatch: expected {self.num_items} or {self.num_items + 1}, got {weight.size(0)}"
            )
        emb_layer.weight.requires_grad = True

    def _build_masks(self, seqs):
        device = seqs.device
        _, seq_len = seqs.size()
        padding_mask = (seqs == 0)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=device).bool()
        return padding_mask, causal_mask

    def _add_position(self, x):
        batch_size, seq_len, _ = x.size()
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        return x + self.pos_emb(positions)

    def get_item_modalities(self, item_ids):
        e_id = self.item_emb(item_ids)
        e_acoustic = self.acoustic_proj(self.acoustic_emb(item_ids))
        e_graph = self.graph_proj(self.graph_emb(item_ids))
        return e_id, e_acoustic, e_graph

    def encode_modalities(self, seqs):
        """
        返回上下文化后的三路表示：
        z_id, z_acoustic, z_graph: [B, L, D]
        """
        padding_mask, causal_mask = self._build_masks(seqs)

        # 三路基础表示
        e_id, e_acoustic, e_graph = self.get_item_modalities(seqs)

        # 加位置编码
        id_embs = self._add_position(e_id)
        acoustic_embs = self._add_position(e_acoustic)
        graph_embs = self._add_position(e_graph)

        # 三流独立上下文化
        id_out = self.id_encoder(
            self.dropout(id_embs),
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )
        acoustic_out = self.acoustic_encoder(
            self.dropout(acoustic_embs),
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )
        graph_out = self.graph_encoder(
            self.dropout(graph_embs),
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
        )
        return id_out, acoustic_out, graph_out

    def forward(self, seqs, return_gate=False):
        """序列编码：输出归一化后的隐状态 [B, L, D]。"""
        id_out, acoustic_out, graph_out = self.encode_modalities(seqs)
        if return_gate:
            fused_out, gate = self.fusion(id_out, acoustic_out, graph_out, return_gate=True)
            return fused_out, gate
        return self.fusion(id_out, acoustic_out, graph_out, return_gate=False)

    def get_item_representation(self, item_ids, return_gate=False):
        """物品综合编码：与 sequence 侧共享同一套融合逻辑。"""
        e_id, e_acoustic, e_graph = self.get_item_modalities(item_ids)
        if return_gate:
            rep, gate = self.fusion(e_id, e_acoustic, e_graph, return_gate=True)
            return rep, gate
        return self.fusion(e_id, e_acoustic, e_graph, return_gate=False)

    def get_last_step_gate(self, seqs):
        """
        返回最后一步的 gate 权重，便于可视化：
        gates: [B, 3]，三列分别是 [ID_gate, Acoustic_gate, Graph_gate]
        """
        _, gate = self.forward(seqs, return_gate=True)
        return gate[:, -1, :]


In [26]:
import torch
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time

print("🌌 准备进行极速异构图消息传递 (LightGCN-style)...")
start_g_time = time.time()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. 将 Mapping 数据转为 Pandas (极速提取边)
# ==========================================
df_artist = artist_item_mapping.to_pandas()
df_album = album_item_mapping.to_pandas()

num_items = num_tracks # 复用之前的总量

# 找出艺人和专辑的最大 ID 以确定矩阵形状
num_artists = df_artist['artist_id'].max()
num_albums = df_album['album_id'].max()

# ==========================================
# 2. 核心魔法函数：构建双向对称归一化稀疏矩阵 D^{-1/2} M D^{-1/2}
# ==========================================
def build_normalized_bipartite_matrix(df, item_col, entity_col, n_items, n_entities):
    items = df[item_col].values
    entities = df[entity_col].values
    data = np.ones(len(items), dtype=np.float32)

    # 构建基础的稀疏共现矩阵 M (Item x Entity)
    M = sp.coo_matrix((data, (items, entities)), shape=(n_items + 1, n_entities + 1))

    # 计算度数 (Degree)
    item_degree = np.array(M.sum(axis=1)).flatten()
    entity_degree = np.array(M.sum(axis=0)).flatten()

    # D^{-1/2}
    d_inv_item = np.power(item_degree, -0.5, where=item_degree > 0)
    d_inv_item[item_degree == 0] = 0.0
    d_inv_entity = np.power(entity_degree, -0.5, where=entity_degree > 0)
    d_inv_entity[entity_degree == 0] = 0.0

    # 矩阵缩放: D_item * M * D_entity
    D_item = sp.diags(d_inv_item)
    D_entity = sp.diags(d_inv_entity)
    M_norm = D_item.dot(M).dot(D_entity).tocoo()

    # 转为 PyTorch GPU 稀疏张量 (极速相乘的底座)
    indices = torch.tensor(np.vstack((M_norm.row, M_norm.col)), dtype=torch.long)
    values = torch.tensor(M_norm.data, dtype=torch.float32)
    shape = M_norm.shape
    return torch.sparse_coo_tensor(indices, values, torch.Size(shape)).to(device)

print("📐 正在构建并归一化 [歌曲-艺人] 与 [歌曲-专辑] 二分图矩阵...")
M_artist = build_normalized_bipartite_matrix(df_artist, 'item_id', 'artist_id', num_items, num_artists)
M_album = build_normalized_bipartite_matrix(df_album, 'item_id', 'album_id', num_items, num_albums)

# ==========================================
# 3. 开始图神经网络消息传递 (Message Passing)
# ==========================================
print("🚀 正在 GPU 上进行图卷积消息传递 (K=2 Hop)...")
graph_dim = 64

# 随机初始化所有物品的结构向量 E_item_0 (用正态分布起手)
E_item = torch.nn.init.normal_(torch.empty(num_items + 1, graph_dim, device=device), std=0.1)

# 我们把第一层的特征保存下来做跳跃连接 (Skip Connection)
E_final = [E_item]

# 进行 2 层图卷积 (2-Hop 刚好能连接 [同艺人] 和 [同专辑] 的不同歌曲)
for layer in range(2):
    # 路径 A：歌曲 -> 艺人 -> 歌曲
    # 1. 艺人吸收他所有歌曲的特征
    E_artist = torch.sparse.mm(M_artist.t(), E_item)
    # 2. 歌曲吸收所属艺人的特征
    E_item_from_artist = torch.sparse.mm(M_artist, E_artist)

    # 路径 B：歌曲 -> 专辑 -> 歌曲
    E_album = torch.sparse.mm(M_album.t(), E_item)
    E_item_from_album = torch.sparse.mm(M_album, E_album)

    # 将两条路径汇聚，更新本轮的物品特征
    E_item = E_item_from_artist + E_item_from_album
    E_final.append(E_item)

# 将每一层聚合的特征加权平均 (LightGCN 的标准 Readout 操作)
graph_vectors_tensor = torch.mean(torch.stack(E_final), dim=0)
graph_vectors = graph_vectors_tensor.cpu().numpy()

print(f"✅ 图谱结构特征 `graph_vectors` 生成完毕！形状: {graph_vectors.shape}")
print(f"   -> 总耗时: {time.time() - start_g_time:.2f} 秒\n")

🌌 准备进行极速异构图消息传递 (LightGCN-style)...
📐 正在构建并归一化 [歌曲-艺人] 与 [歌曲-专辑] 二分图矩阵...
🚀 正在 GPU 上进行图卷积消息传递 (K=2 Hop)...
✅ 图谱结构特征 `graph_vectors` 生成完毕！形状: (9390624, 64)
   -> 总耗时: 9.92 秒



In [27]:
# 确保之前的 FDSA_Graph_Fusion 类已经定义并运行
print("🛠️ 正在初始化改进版图序列混合网络（共享门控融合头 + 三流独立编码器）...")

model_graph = FDSA_Graph_Fusion(
    num_items=num_tracks,
    item_dim=64,
    acoustic_dim=128,
    graph_dim=64,
    pretrained_acoustic=track_vectors,
    pretrained_graph=graph_vectors,  # 🌟 注入刚刚光速炼成的图谱特征！
    num_heads=4,
    num_layers=2
).to(device)

print("✅ 改进版 FDSA_Graph_Fusion 初始化完毕！共享融合头已生效。")

🛠️ 正在初始化改进版图序列混合网络（共享门控融合头 + 三流独立编码器）...
✅ 改进版 FDSA_Graph_Fusion 初始化完毕！共享融合头已生效。


# 综合对比分析

In [28]:
# 加载模型
from huggingface_hub import snapshot_download
from huggingface_hub import login
login()

snapshot_download(
    repo_id="Pcpp/music-transformer",
    local_dir="music-transformer",
    resume_download=True,
    max_workers=8
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

'/content/music-transformer'

In [29]:
import torch
import itertools

print("🛠️ 正在修复：重新计算全局热度统计 (track_counts)...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. 展平训练集序列，极速统计频次
flat_items = list(itertools.chain.from_iterable(train_seqs.values()))
flat_items_tensor = torch.tensor(flat_items, dtype=torch.long, device=device)
track_counts = torch.bincount(flat_items_tensor, minlength=num_tracks + 1)

# 排除 padding (0) 的干扰
track_counts[0] = 0

# 2. 重新提取热门候选集
POOL_SIZE = min(50000, num_tracks)
top_popular_tracks = torch.topk(track_counts, POOL_SIZE).indices

print("✅ track_counts 补全成功！")

🛠️ 正在修复：重新计算全局热度统计 (track_counts)...
✅ track_counts 补全成功！


In [30]:
# ==========================================
# 0. 防护性补全：测试集 (Test Loader) 与全局属性
# ==========================================
# 重新计算热度和掩码 (防止重启丢失)
flat_items = list(itertools.chain.from_iterable(train_seqs.values()))
flat_items_tensor = torch.tensor(flat_items, dtype=torch.long, device=device)
track_counts = torch.bincount(flat_items_tensor, minlength=num_tracks + 1)
track_counts[0] = 0

POOL_SIZE = min(50000, num_tracks)
top_popular_tracks = torch.topk(track_counts, POOL_SIZE).indices

track_probs = (track_counts / track_counts.sum())
track_novelty = -torch.log2(track_probs + 1e-9)

head_indices = torch.topk(track_counts, min(5000, num_tracks)).indices
is_head_item = torch.zeros(num_tracks + 1, dtype=torch.bool, device=device)
is_head_item[head_indices] = True

# 重新定义 Test DataLoader
class TestSequenceDataset(Dataset):
    def __init__(self, test_seqs, max_len=50):
        self.uids = list(test_seqs.keys())
        self.seqs = [test_seqs[uid] for uid in self.uids]
        self.max_len = max_len

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, idx):
        seq = self.seqs[idx]
        target = seq[-1]
        hist = seq[:-1][-self.max_len:]
        pad_len = self.max_len - len(hist)
        hist = [0] * pad_len + hist
        return torch.tensor(hist, dtype=torch.long), torch.tensor(target, dtype=torch.long)

if 'MAX_SEQ_LEN' not in locals():
    MAX_SEQ_LEN = 50

test_dataset = TestSequenceDataset(test_seqs, max_len=MAX_SEQ_LEN)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [31]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

print("🔍 正在进行极其硬核的【流行度去偏】分层评估 (三模型全量对比)...")

# ==========================================
# 1. 划分 Head, Torso, Tail 集合
# ==========================================
# 按照热度降序排列的 Track IDs
sorted_tracks = torch.argsort(track_counts, descending=True).cpu().numpy()

num_total = len(sorted_tracks)
head_threshold = int(0.05 * num_total)
torso_threshold = int(0.20 * num_total)

head_set = set(sorted_tracks[:head_threshold])
torso_set = set(sorted_tracks[head_threshold:torso_threshold])
tail_set = set(sorted_tracks[torso_threshold:])

# ==========================================
# 2. 准备分层统计容器
# ==========================================
# 分别记录各个区间的总样本数和命中数
strata_results = {
    'ID_Only': {'Head': [0,0], 'Torso': [0,0], 'Tail': [0,0]}, # [hit_count, total_count]
    'FDSA': {'Head': [0,0], 'Torso': [0,0], 'Tail': [0,0]},
    'FDSA_Graph': {'Head': [0,0], 'Torso': [0,0], 'Tail': [0,0]}
}

# ==========================================
# 3. 加载模型权重
# ==========================================

# model_id_only = torch.load("music-transformer/best_model_id_only.pth")
# model = torch.load("music-transformer/best_model_shared_gated_pro.pth")
# model_graph = torch.load("music-transformer/best_model_graph.pth")

# 1. 纯 ID 模型
model_id_only.load_state_dict(torch.load("music-transformer/best_model_id_only.pth"))
model_id_only.eval()

# 2. 🌟 双流模型 (假设你之前的模型实例名为 model，权重保存在 best_model_pro.pth)
# 如果名字不同，请自行修改为你保存的真实文件名
model.load_state_dict(torch.load("music-transformer/best_model_shared_gated_pro.pth"))
model.eval()

# 3. 三流终极模型
model_graph.load_state_dict(torch.load("music-transformer/best_model_graph.pth"))
model_graph.eval()

print("🚀 开始扫描测试集，进行分层命中判定...")

with torch.no_grad():
    for x, target in tqdm(test_loader, desc="分层评测进度"):
        x, target = x.to(device), target.to(device)
        batch_size = x.size(0)
        target_np = target.cpu().numpy()

        # 准备候选集 (1 正 + 999 负)
        pos_tids = target.unsqueeze(1)
        neg_idx = torch.randint(0, POOL_SIZE, (batch_size, 999), device=device)
        neg_candidates = top_popular_tracks[neg_idx]
        candidates_tids = torch.cat([pos_tids, neg_candidates], dim=1)

        # --- A. 评估纯 ID 模型 ---
        h_id = model_id_only(x)[:, -1, :].unsqueeze(1)
        cand_id = model_id_only.get_item_representation(candidates_tids)
        scores_id = (h_id * cand_id).sum(dim=-1)
        _, top100_id = torch.topk(scores_id, k=100, dim=1)
        hits_id = (top100_id == 0).float().sum(dim=1).cpu().numpy()

        # --- B. 🌟 评估 FDSA 双流模型 ---
        h_fdsa = model(x)[:, -1, :].unsqueeze(1)
        cand_fdsa = model.get_item_representation(candidates_tids)
        scores_fdsa = (h_fdsa * cand_fdsa).sum(dim=-1)
        _, top100_fdsa = torch.topk(scores_fdsa, k=100, dim=1)
        hits_fdsa = (top100_fdsa == 0).float().sum(dim=1).cpu().numpy()

        # --- C. 评估三流终极模型 ---
        h_g = model_graph(x)[:, -1, :].unsqueeze(1)
        cand_g = model_graph.get_item_representation(candidates_tids)
        scores_g = (h_g * cand_g).sum(dim=-1)
        _, top100_g = torch.topk(scores_g, k=100, dim=1)
        hits_g = (top100_g == 0).float().sum(dim=1).cpu().numpy()

        # --- D. 分发结果到各个阶层 ---
        for i in range(batch_size):
            t = target_np[i]
            if t in head_set: group = 'Head'
            elif t in torso_set: group = 'Torso'
            else: group = 'Tail'

            # 统计总数
            strata_results['ID_Only'][group][1] += 1
            strata_results['FDSA'][group][1] += 1
            strata_results['FDSA_Graph'][group][1] += 1

            # 统计命中数
            strata_results['ID_Only'][group][0] += hits_id[i]
            strata_results['FDSA'][group][0] += hits_fdsa[i]
            strata_results['FDSA_Graph'][group][0] += hits_g[i]

# ==========================================
# 4. 打印分层报告
# ==========================================
print("\n" + "="*65)
print("🏆 【深度洞察】三大模型流行度分层抗压测试 (Recall@100)")
print("="*65)
groups = ['Head', 'Torso', 'Tail']
for g in groups:
    count = strata_results['ID_Only'][g][1]
    if count == 0: continue
    rec_id = strata_results['ID_Only'][g][0] / count
    rec_fdsa = strata_results['FDSA'][g][0] / count
    rec_g = strata_results['FDSA_Graph'][g][0] / count

    print(f"🎯 【{g} 组】(占测试集总样本: {count} 个)")
    print(f"   - 纯 ID 基线 : {rec_id:.4f}")
    print(f"   - FDSA (双流): {rec_fdsa:.4f}")
    print(f"   - FDSA-Graph : {rec_g:.4f}  <-- 相较纯 ID 提升了 {(rec_g - rec_id)*100:.2f}%")
    print("-" * 65)

🔍 正在进行极其硬核的【流行度去偏】分层评估 (三模型全量对比)...
🚀 开始扫描测试集，进行分层命中判定...



分层评测进度: 100%|██████████| 194/194 [00:11<00:00, 16.97it/s]


🏆 【深度洞察】三大模型流行度分层抗压测试 (Recall@100)
🎯 【Head 组】(占测试集总样本: 95734 个)
   - 纯 ID 基线 : 0.8062
   - FDSA (双流): 0.8033
   - FDSA-Graph : 0.8174  <-- 相较纯 ID 提升了 1.12%
-----------------------------------------------------------------
🎯 【Torso 组】(占测试集总样本: 3128 个)
   - 纯 ID 基线 : 0.4102
   - FDSA (双流): 0.7762
   - FDSA-Graph : 0.7957  <-- 相较纯 ID 提升了 38.55%
-----------------------------------------------------------------
🎯 【Tail 组】(占测试集总样本: 67 个)
   - 纯 ID 基线 : 0.7313
   - FDSA (双流): 0.7463
   - FDSA-Graph : 0.7612  <-- 相较纯 ID 提升了 2.99%
-----------------------------------------------------------------


In [32]:
import torch
import numpy as np
from tqdm import tqdm

print("🔍 启动【无掩码纯净流】冷启动极限抗压测试 (剥离 Padding 干扰)...")

# 准备真实容器
true_cold_results = {
    'ID_Only': {'Short (<10)': [0,0], 'Medium (10-30)': [0,0]},
    'FDSA': {'Short (<10)': [0,0], 'Medium (10-30)': [0,0]},
    'FDSA_Graph': {'Short (<10)': [0,0], 'Medium (10-30)': [0,0]}
}

model_id_only.eval()
model.eval()
model_graph.eval()

# 提取所有的短序列和中等序列用户进行逐一(Batch=1)精准打击
cold_users_uids = [uid for uid, seq in test_seqs.items() if len(seq) <= 31]

print(f"🚀 共锁定 {len(cold_users_uids)} 名冷启动/中度活跃用户，开始逐一穿透扫描...")

with torch.no_grad():
    for uid in tqdm(cold_users_uids, desc="真实能力探测"):
        seq = test_seqs[uid]
        target = seq[-1]

        # 🌟 核心：绝对不加任何 0 Padding！保持最原始的纯净序列
        hist = seq[:-1]
        seq_len = len(hist)

        if seq_len == 0: continue # 极个别异常数据跳过

        if seq_len < 10: group = 'Short (<10)'
        else: group = 'Medium (10-30)'

        # 转换为 Batch=1 的 Tensor [1, L]
        x_raw = torch.tensor([hist], dtype=torch.long, device=device)
        pos_tids = torch.tensor([[target]], device=device)

        # 负采样
        neg_idx = torch.randint(0, POOL_SIZE, (1, 999), device=device)
        neg_candidates = top_popular_tracks[neg_idx]
        candidates_tids = torch.cat([pos_tids, neg_candidates], dim=1)

        # ==========================================
        # 释放模型的真实战力 (无需处理 NaN)
        # ==========================================
        # A. 纯 ID 模型
        h_id = model_id_only(x_raw)[:, -1, :].unsqueeze(1)
        cand_id = model_id_only.get_item_representation(candidates_tids)
        scores_id = (h_id * cand_id).sum(dim=-1)
        hit_id = (torch.topk(scores_id, k=10, dim=1)[1] == 0).float().sum().item()

        # B. FDSA 双流模型
        h_fdsa = model(x_raw)[:, -1, :].unsqueeze(1)
        cand_fdsa = model.get_item_representation(candidates_tids)
        scores_fdsa = (h_fdsa * cand_fdsa).sum(dim=-1)
        hit_fdsa = (torch.topk(scores_fdsa, k=10, dim=1)[1] == 0).float().sum().item()

        # C. FDSA-Graph 三流终极模型
        h_g = model_graph(x_raw)[:, -1, :].unsqueeze(1)
        cand_g = model_graph.get_item_representation(candidates_tids)
        scores_g = (h_g * cand_g).sum(dim=-1)
        hit_g = (torch.topk(scores_g, k=10, dim=1)[1] == 0).float().sum().item()

        # 录入成绩
        true_cold_results['ID_Only'][group][1] += 1
        true_cold_results['FDSA'][group][1] += 1
        true_cold_results['FDSA_Graph'][group][1] += 1

        true_cold_results['ID_Only'][group][0] += hit_id
        true_cold_results['FDSA'][group][0] += hit_fdsa
        true_cold_results['FDSA_Graph'][group][0] += hit_g

# ==========================================
# 打印浴火重生的战报
# ==========================================
print("\n" + "="*65)
print("🏆 【真实极限】剥离 Padding 后的用户冷启动真实战力 (Recall@10)")
print("="*65)
for g in ['Short (<10)', 'Medium (10-30)']:
    count = true_cold_results['ID_Only'][g][1]
    if count == 0: continue
    rec_id = true_cold_results['ID_Only'][g][0] / count
    rec_fdsa = true_cold_results['FDSA'][g][0] / count
    rec_g = true_cold_results['FDSA_Graph'][g][0] / count

    print(f"🎯 【{g} 组】(真实无损用户数: {count} 人)")
    print(f"   - 纯 ID 基线 : {rec_id:.4f}")
    print(f"   - FDSA (双流): {rec_fdsa:.4f}")
    print(f"   - FDSA-Graph : {rec_g:.4f}")
    print("-" * 65)

🔍 启动【无掩码纯净流】冷启动极限抗压测试 (剥离 Padding 干扰)...
🚀 共锁定 4252 名冷启动/中度活跃用户，开始逐一穿透扫描...


真实能力探测: 100%|██████████| 4252/4252 [00:50<00:00, 84.26it/s]


🏆 【真实极限】剥离 Padding 后的用户冷启动真实战力 (Recall@10)
🎯 【Short (<10) 组】(真实无损用户数: 1094 人)
   - 纯 ID 基线 : 0.3967
   - FDSA (双流): 0.3967
   - FDSA-Graph : 0.4022
-----------------------------------------------------------------
🎯 【Medium (10-30) 组】(真实无损用户数: 3158 人)
   - 纯 ID 基线 : 0.4278
   - FDSA (双流): 0.4281
   - FDSA-Graph : 0.4436
-----------------------------------------------------------------


In [33]:
import torch

print("📚 正在探查 Hugging Face 数据集结构...")

# 1. 查看数据结构 (打印第一条数据看看长什么样)
print("\n[Album Mapping 样例]:", album_item_mapping[0])
print("[Artist Mapping 样例]:", artist_item_mapping[0])

# 2. 极速构建双重映射字典
print("\n⚙️ 正在构建 O(1) 复杂度的全局哈希映射字典...")

item_to_album = {}
# 假设字段名为 'item_id' 和 'album_id' (如果报错，请根据上面的打印结果修改字段名)
for row in album_item_mapping:
    item_to_album[int(row['item_id'])] = int(row['album_id'])

item_to_artist = {}
# 假设字段名为 'item_id' 和 'artist_id'
for row in artist_item_mapping:
    item_to_artist[int(row['item_id'])] = int(row['artist_id'])

print(f"✅ 构建完成！共收录了 {len(item_to_album)} 条专辑映射，{len(item_to_artist)} 条艺人映射。")

📚 正在探查 Hugging Face 数据集结构...

[Album Mapping 样例]: {'album_id': 1, 'item_id': 1491131}
[Artist Mapping 样例]: {'artist_id': 1, 'item_id': 953587}

⚙️ 正在构建 O(1) 复杂度的全局哈希映射字典...
✅ 构建完成！共收录了 8653783 条专辑映射，9270506 条艺人映射。


In [65]:
import random

print("🔍 正在启动【双重图谱语义】微观案例解析 (Case Study)...")

# 升级版信息获取函数
def get_track_info(tid):
    tid_int = int(tid)
    artist_id = item_to_artist.get(tid_int, "未知艺人")
    album_id = item_to_album.get(tid_int, "未知专辑")
    return f"[Track_{tid_int:>7}] (Artist: {artist_id:>6} | Album: {album_id:>7})"

# 寻找具有代表性的中等活跃目标用户 (10~20首)
candidate_uids = [uid for uid, seq in test_seqs.items() if 10 <= len(seq) <= 20]
target_uid = random.choice(candidate_uids)

seq = test_seqs[target_uid]
target_item = seq[-1]
hist_items = seq[:-1]

# 获取 Ground Truth 的高阶图谱信息
target_artist = item_to_artist.get(int(target_item), "未知艺人")
target_album = item_to_album.get(int(target_item), "未知专辑")

print("\n" + "="*85)
print(f"👤 【测试用户 UID: {target_uid}】的结构化听歌脉络")
print("="*85)
print("🎵 [用户的历史听歌序列 (Input Context)]:")
for i, tid in enumerate(hist_items):
    print(f"   {i+1:>2}. {get_track_info(tid)}")
print("-" * 85)
print(f"🎯 [用户接下来真实点击的歌曲 (Ground Truth)]:")
print(f"   👉 {get_track_info(target_item)}")
print("="*85)

# 准备该用户的输入 Tensor
x_raw = torch.tensor([hist_items], dtype=torch.long, device=device)
all_items = torch.arange(1, num_tracks + 1, device=device)

with torch.no_grad():
    print("\n🧠 各大模型的 Top-5 推荐大脑在想什么？")

    # --- A. 纯 ID 模型 ---
    h_id = model_id_only(x_raw)[:, -1, :]
    cand_id = model_id_only.get_item_representation(all_items)
    scores_id = (h_id * cand_id).sum(dim=-1)
    top5_id = torch.topk(scores_id, k=5).indices + 1

    print("\n 1. ID-Only Baseline (纯 ID 协同过滤):")
    for i, tid in enumerate(top5_id.cpu().numpy()):
        match = "🎯 命中 Track!" if tid == target_item else ""
        art_match = "✨ 命中 Artist!" if item_to_artist.get(int(tid)) == target_artist and tid != target_item else ""
        alb_match = "💿 命中 Album!" if item_to_album.get(int(tid)) == target_album and tid != target_item else ""
        print(f"   Top-{i+1}: {get_track_info(tid)} {match}{alb_match}{art_match}")

    # --- B. FDSA 双流模型 ---
    h_fdsa = model(x_raw)[:, -1, :]
    cand_fdsa = model.get_item_representation(all_items)
    scores_fdsa = (h_fdsa * cand_fdsa).sum(dim=-1)
    top5_fdsa = torch.topk(scores_fdsa, k=5).indices + 1

    print("\n 2. FDSA Dual-stream (融合 128维声学特征):")
    for i, tid in enumerate(top5_fdsa.cpu().numpy()):
        match = "🎯 命中 Track!" if tid == target_item else ""
        art_match = "✨ 命中 Artist!" if item_to_artist.get(int(tid)) == target_artist and tid != target_item else ""
        alb_match = "💿 命中 Album!" if item_to_album.get(int(tid)) == target_album and tid != target_item else ""
        print(f"   Top-{i+1}: {get_track_info(tid)} {match}{alb_match}{art_match}")

    # --- C. FDSA-Graph 三流终极模型 ---
    h_g = model_graph(x_raw)[:, -1, :]
    cand_g = model_graph.get_item_representation(all_items)
    scores_g = (h_g * cand_g).sum(dim=-1)
    top5_g = torch.topk(scores_g, k=5).indices + 1

    print("\n 3. FDSA-Graph Tri-stream (图谱协同 + 声学多模态):")
    for i, tid in enumerate(top5_g.cpu().numpy()):
        match = "🎯 命中 Track!" if tid == target_item else ""
        art_match = "✨ 命中 Artist!" if item_to_artist.get(int(tid)) == target_artist and tid != target_item else ""
        alb_match = "💿 命中 Album!" if item_to_album.get(int(tid)) == target_album and tid != target_item else ""
        print(f"   Top-{i+1}: {get_track_info(tid)} {match}{alb_match}{art_match}")
print("\n" + "="*85)

🔍 正在启动【双重图谱语义】微观案例解析 (Case Study)...

👤 【测试用户 UID: 441420】的结构化听歌脉络
🎵 [用户的历史听歌序列 (Input Context)]:
    1. [Track_9175838] (Artist: 130879 | Album:  226131)
    2. [Track_3694666] (Artist: 520570 | Album: 3239693)
    3. [Track_7602951] (Artist: 634525 | Album: 1635853)
    4. [Track_1348436] (Artist: 634525 | Album: 1635853)
    5. [Track_ 844851] (Artist: 634525 | Album: 2433573)
    6. [Track_1754260] (Artist: 644542 | Album:  232845)
    7. [Track_1023158] (Artist: 644123 | Album:  645377)
    8. [Track_1266625] (Artist: 838787 | Album: 2146554)
    9. [Track_7897953] (Artist: 913489 | Album: 1974220)
   10. [Track_7419818] (Artist: 1264934 | Album: 2158930)
   11. [Track_1380298] (Artist: 436018 | Album: 1644912)
-------------------------------------------------------------------------------------
🎯 [用户接下来真实点击的歌曲 (Ground Truth)]:
   👉 [Track_3009135] (Artist: 1242819 | Album: 2377497)

🧠 各大模型的 Top-5 推荐大脑在想什么？

 1. ID-Only Baseline (纯 ID 协同过滤):
   Top-1: [Track_6144533] (Artist: 6441